In [2]:
import re

def parse_expr(expr):
    """Convert expression into a nested structure."""
    tokens = re.findall(r'[A-Za-z_]\w*|[(),]', expr.replace(" ", ""))
    pos = 0

    def parse():
        nonlocal pos
        if pos >= len(tokens):
            raise ValueError("Incomplete expression")

        name = tokens[pos]
        pos += 1

        if pos < len(tokens) and tokens[pos] == '(':
            pos += 1
            args = []

            while True:
                args.append(parse())

                if pos >= len(tokens):
                    raise ValueError("Missing closing parenthesis")

                if tokens[pos] == ',':
                    pos += 1
                elif tokens[pos] == ')':
                    pos += 1
                    break
                else:
                    raise ValueError("Invalid expression syntax")

            return (name, args)

        return name

    result = parse()

    if pos != len(tokens):
        raise ValueError("Unexpected extra input")

    return result


def is_variable(term):
    """Variables are single lowercase letters."""
    return (
        isinstance(term, str)
        and len(term) == 1
        and term.islower()
        and term.isalpha()
    )


def format_term(term):
    if isinstance(term, str):
        return term

    name, args = term
    return name + "(" + ", ".join(
        format_term(a) for a in args
    ) + ")"


def apply_substitution(term, subst):
    if isinstance(term, str):
        while term in subst:
            term = subst[term]
            if isinstance(term, tuple):
                return apply_substitution(term, subst)
        return term

    name, args = term
    return (
        name,
        [apply_substitution(a, subst) for a in args]
    )


def occurs_check(var, term, subst):
    term = apply_substitution(term, subst)

    if term == var:
        return True

    if isinstance(term, tuple):
        return any(
            occurs_check(var, arg, subst)
            for arg in term[1]
        )

    return False


def unify_terms(t1, t2, subst):
    t1 = apply_substitution(t1, subst)
    t2 = apply_substitution(t2, subst)

    # Both terms are already identical
    if t1 == t2:
        return True, ""

    # First term is a variable
    if is_variable(t1):
        if occurs_check(t1, t2, subst):
            return False, (
                f"Occurs check failed: variable {t1} "
                f"appears inside {format_term(t2)}."
            )

        subst[t1] = t2
        return True, ""

    # Second term is a variable
    if is_variable(t2):
        if occurs_check(t2, t1, subst):
            return False, (
                f"Occurs check failed: variable {t2} "
                f"appears inside {format_term(t1)}."
            )

        subst[t2] = t1
        return True, ""

    # Both terms are compound expressions
    if isinstance(t1, tuple) and isinstance(t2, tuple):

        if t1[0] != t2[0]:
            return False, (
                f"Function symbols differ: '{t1[0]}' "
                f"and '{t2[0]}'."
            )

        if len(t1[1]) != len(t2[1]):
            return False, (
                f"Different numbers of arguments in "
                f"'{format_term(t1)}' and "
                f"'{format_term(t2)}'."
            )

        for a, b in zip(t1[1], t2[1]):
            success, reason = unify_terms(a, b, subst)
            if not success:
                return False, reason

        return True, ""

    # Two different constants cannot be unified
    return False, (
        f"Conflicting constants or terms: "
        f"'{format_term(t1)}' and '{format_term(t2)}' "
        f"are different and cannot be substituted."
    )


def unify(expr1, expr2):
    try:
        t1 = parse_expr(expr1)
        t2 = parse_expr(expr2)
    except ValueError as error:
        return None, f"Invalid expression: {error}", {}

    # Expressions must have matching predicate symbols
    if not isinstance(t1, tuple) or not isinstance(t2, tuple):
        return None, (
            "Both expressions must contain predicate symbols "
            "with arguments, such as P(x)."
        ), {}

    if t1[0] != t2[0]:
        return None, (
            f"Predicate symbols differ: '{t1[0]}' and "
            f"'{t2[0]}'."
        ), {}

    if len(t1[1]) != len(t2[1]):
        return None, (
            f"Different numbers of arguments: "
            f"'{t1[0]}' has {len(t1[1])}, while "
            f"'{t2[0]}' has {len(t2[1])}."
        ), {}

    subst = {}

    for a, b in zip(t1[1], t2[1]):
        success, reason = unify_terms(a, b, subst)

        if not success:
            return None, reason, subst

    return True, "", subst


def main():
    print("FIRST ORDER LOGIC - UNIFICATION")
    print("--------------------------------")

    expr1 = input("Enter first expression: ")
    expr2 = input("Enter second expression: ")

    success, reason, subst = unify(expr1, expr2)

    if success is None:
        print("\nUnification failed!")
        print("Reason:", reason)

    elif success is False:
        print("\nUnification failed!")
        print("Reason:", reason)

    elif not subst:
        print("\nUnification successful!")
        print("Expressions are already identical.")
        print("Substitution: NIL")

    else:
        print("\nUnification successful!")
        print("Substitution:")

        for var, value in subst.items():
            print(f"{var} = {format_term(value)}")

        print("\nUnified Expression:")
        print(format_term(
            apply_substitution(parse_expr(expr1), subst)
        ))


if __name__ == "__main__":
    main()



FIRST ORDER LOGIC - UNIFICATION
--------------------------------
Enter first expression: p(b,x,f(g(z)))
Enter second expression: p(z,f(y),f(y))

Unification successful!
Substitution:
b = z
x = f(y)
y = g(z)

Unified Expression:
p(z, f(g(z)), f(g(z)))
